# 02 · Colombia and the Vibecession

| | |
|---|---|
| **Question** | Do Colombians feel worse about the economy than inflation, unemployment, interest rates and markets would predict? |
| **Data** | FRED (OECD series for Colombia: consumer confidence, inflation, unemployment, interest rate, exchange rate, stock index). Cached copies in `data/` |
| **Skills** | Merging monthly series, regression on time series, reading residuals as the gap between vibes and fundamentals |
| **Runs in** | Colab: yes. JupyterLite: yes (uses the cached data) |

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ericvd-ucb/Colombia-26-NB/blob/main/notebooks/02_colombia_vibecession.ipynb) [![Open in JupyterLite](https://jupyterlite.rtfd.io/en/latest/_static/badge.svg)](https://ericvd-ucb.github.io/Colombia-26-NB/lab/index.html?path=notebooks/02_colombia_vibecession.ipynb)

*Eric Van Dusen, UC Berkeley College of Computing, Data Science, and Society. Prepared for students in Colombia, October 2026.*

In [ ]:
# Setup: run this cell first. It works in Google Colab, JupyterLite, or on your own computer.
import os, sys
IN_COLAB = "google.colab" in sys.modules
IN_JUPYTERLITE = sys.platform == "emscripten"

# Saved data lives in the repo's data/ folder; in Colab we download it from GitHub when needed
DATA_URL = "https://raw.githubusercontent.com/ericvd-ucb/Colombia-26-NB/main/data/"
DATA_DIR = "../data" if os.path.isdir("../data") else "data"
os.makedirs(DATA_DIR, exist_ok=True)

def data_path(filename):
    """Local path to a file from the repo's data/ folder, downloading it first if it isn't here."""
    local = os.path.join(DATA_DIR, filename)
    if not os.path.exists(local):
        import urllib.request
        urllib.request.urlretrieve(DATA_URL + filename, local)
    return local

if IN_JUPYTERLITE:
    %pip install -q pandas statsmodels

import pandas as pd

def fred_csv(url):
    """Read a FRED series by URL. Saves a copy to data/; if the live download fails, uses that copy."""
    series_id = url.split("id=")[-1]
    filename = f"fred_{series_id}.csv"
    try:
        df = pd.read_csv(url)
        df.to_csv(os.path.join(DATA_DIR, filename), index=False)
        return df
    except Exception as err:
        print(f"Live download of {series_id} unavailable ({type(err).__name__}); using saved copy.")
        return pd.read_csv(data_path(filename))

# Colombia and the Vibecession

In the United States, the word **vibecession** was used to describe a period when the economy looked fairly strong in conventional macroeconomic data, but consumers still felt bad about it.

Colombia gives us a useful extension of that question.

Consumer confidence in Colombia has moved dramatically over the last several years. It collapsed during COVID, fell again during the 2021 protests, became deeply negative during the inflation shock of 2022–23, and then rose strongly in 2025–26.

Our question is:

> **When Colombians feel good or bad about the economy, how much can we explain with macroeconomic conditions?**

We will use a simple model to estimate what consumer confidence *should* look like given inflation, unemployment, interest rates, and the exchange rate. The difference between actual and predicted confidence will be our **vibe gap**.


## 1. Consumer confidence in Colombia

Colombia's consumer confidence survey is produced by **Fedesarrollo** through the *Encuesta de Opinión del Consumidor*. The OECD publishes a harmonized, seasonally adjusted version of the series, and FRED makes that OECD series easy to download.

The measure is a **percentage balance**. Positive values mean optimistic responses outweigh pessimistic responses. Negative values mean the opposite.

We can download the data directly as a CSV. No FRED API key is needed.


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

In [ ]:
url = "https://fred.stlouisfed.org/graph/fredgraph.csv?id=COLCSCICP02STSAM"

confidence = fred_csv(url)
confidence.head()

In [ ]:
confidence = confidence.rename(
    columns={"COLCSCICP02STSAM": "Consumer Confidence"}
)

confidence["observation_date"] = pd.to_datetime(confidence["observation_date"])
confidence.tail()

In [ ]:
confidence.plot(
    x="observation_date",
    y="Consumer Confidence",
    figsize=(12, 5),
    legend=False
)

plt.axhline(0, linewidth=1)
plt.title("Consumer Confidence in Colombia")
plt.ylabel("Percentage balance")
plt.xlabel("")
plt.show()

### What do you notice?

Look for periods when confidence changes very quickly.

Some dates worth keeping in mind:

- **2020:** COVID-19 and the shutdown
- **April–May 2021:** tax reform protests, national strike, road blockades, and political uncertainty
- **2022–23:** high inflation and sharply higher interest rates
- **2025–26:** a striking recovery in consumer confidence

Before we build a model, write down your hypothesis:

**Which macroeconomic variable do you think will be most closely related to Colombian consumer confidence? Why?**


## 2. Possible drivers of the vibes

We will start with four variables that households are likely to notice directly:

1. **Inflation** — are prices rising quickly?
2. **Unemployment** — how difficult is it to find work?
3. **Interest rates** — how expensive is credit?
4. **Exchange rate** — how many Colombian pesos does it take to buy one U.S. dollar?

These are all monthly series, which makes them relatively easy to compare with monthly consumer confidence.


### Inflation

For inflation we use the OECD measure of the year-over-year change in Colombia's Consumer Price Index.

**Note:** this OECD/FRED CPI series currently ends earlier than some of our other series. Later we may replace it with a direct DANE download so that the model can extend all the way through 2026.


In [ ]:
inflation_url = "https://fred.stlouisfed.org/graph/fredgraph.csv?id=COLCPALTT01GYM"

inflation = fred_csv(inflation_url)
inflation = inflation.rename(
    columns={"COLCPALTT01GYM": "Inflation"}
)

inflation["observation_date"] = pd.to_datetime(inflation["observation_date"])
inflation.tail()

In [ ]:
inflation.plot(
    x="observation_date",
    y="Inflation",
    figsize=(12, 4),
    legend=False
)

plt.title("Inflation in Colombia")
plt.ylabel("Percent change from one year earlier")
plt.xlabel("")
plt.show()

### Unemployment

We use the OECD seasonally adjusted monthly unemployment rate for people age 15 and older.


In [ ]:
unemployment_url = "https://fred.stlouisfed.org/graph/fredgraph.csv?id=COLLRUNTTTTSTSAM"

unemployment = fred_csv(unemployment_url)
unemployment = unemployment.rename(
    columns={"COLLRUNTTTTSTSAM": "Unemployment"}
)

unemployment["observation_date"] = pd.to_datetime(unemployment["observation_date"])
unemployment.tail()

In [ ]:
unemployment.plot(
    x="observation_date",
    y="Unemployment",
    figsize=(12, 4),
    legend=False
)

plt.title("Unemployment in Colombia")
plt.ylabel("Percent")
plt.xlabel("")
plt.show()

### Interest rates

For borrowing conditions we use the OECD 3-month interbank interest rate.

This is not exactly the Banco de la República policy rate, but it is a useful monthly measure of short-term Colombian interest rates and closely reflects monetary conditions.


In [ ]:
interest_url = "https://fred.stlouisfed.org/graph/fredgraph.csv?id=COLIR3TIB01STM"

interest = fred_csv(interest_url)
interest = interest.rename(
    columns={"COLIR3TIB01STM": "Interest Rate"}
)

interest["observation_date"] = pd.to_datetime(interest["observation_date"])
interest.tail()

In [ ]:
interest.plot(
    x="observation_date",
    y="Interest Rate",
    figsize=(12, 4),
    legend=False
)

plt.title("Short-Term Interest Rates in Colombia")
plt.ylabel("Percent")
plt.xlabel("")
plt.show()

### Exchange rate

The Colombian peso can also affect household perceptions. A weaker peso makes imported goods more expensive and can be interpreted as a sign of economic instability.

This series measures **Colombian pesos per U.S. dollar**, so a higher number means a weaker peso.


In [ ]:
exchange_url = "https://fred.stlouisfed.org/graph/fredgraph.csv?id=COLCCUSMA02STM"

exchange = fred_csv(exchange_url)
exchange = exchange.rename(
    columns={"COLCCUSMA02STM": "Pesos per Dollar"}
)

exchange["observation_date"] = pd.to_datetime(exchange["observation_date"])
exchange.tail()

In [ ]:
exchange.plot(
    x="observation_date",
    y="Pesos per Dollar",
    figsize=(12, 4),
    legend=False
)

plt.title("Colombian Peso per U.S. Dollar")
plt.ylabel("COP per USD")
plt.xlabel("")
plt.show()

### Colombian stock market

Financial markets may also affect economic perceptions. We use the OECD/FRED broad share-price index for Colombia.

The series is monthly and indexed so that 2015 = 100.


In [ ]:
stock_url = "https://fred.stlouisfed.org/graph/fredgraph.csv?id=COLSPASTT01IXOBM"

stocks = fred_csv(stock_url)
stocks = stocks.rename(
    columns={"COLSPASTT01IXOBM": "Stock Market"}
)

stocks["observation_date"] = pd.to_datetime(stocks["observation_date"])
stocks.tail()

In [ ]:
stocks.plot(
    x="observation_date",
    y="Stock Market",
    figsize=(12, 4),
    legend=False
)

plt.title("Colombian Stock Market")
plt.ylabel("Index, 2015 = 100")
plt.xlabel("")
plt.show()

## 3. Put the data together

Each DataFrame has one row per month. FRED calls the date column `observation_date`. We rename each value column before merging so that every series has a unique name.

Because unemployment begins in 2007, our combined dataset will begin there. Because the inflation series currently ends earlier than the others, the complete-case dataset will end at the last available inflation observation.


In [ ]:
data = confidence.merge(
    inflation,
    on="observation_date",
    how="outer"
)

data = data.merge(
    unemployment,
    on="observation_date",
    how="outer"
)

data = data.merge(
    interest,
    on="observation_date",
    how="outer"
)

data = data.merge(
    exchange,
    on="observation_date",
    how="outer"
)

data = data.merge(
    stocks,
    on="observation_date",
    how="outer"
)

data.tail()

In [ ]:
data = data.sort_values("observation_date")

model_data = data.dropna().copy()

model_data.head()

### What time period do we actually have?

When we merge time series, the usable sample is determined by the variable with the **latest starting date** and the variable with the **earliest ending date**.

The outer merge above preserves every observation from every series. But the regression needs a row where **all variables are present**, so `dropna()` creates our complete-case dataset.

Let's check the resulting time period rather than assuming it.


In [ ]:
print("First complete month:", model_data["observation_date"].min())
print("Last complete month: ", model_data["observation_date"].max())
print("Number of months:    ", len(model_data))

With the current FRED/OECD series, the complete-case dataset begins when the unemployment series begins and ends when the inflation series ends.

That gives us roughly:

- **Training period:** 2007–2019
- **Test period:** 2020–2025

This is enough to study the COVID shock, the 2021 protests, and the 2022–23 inflation episode. But it still misses part of the 2025–26 surge in confidence because the OECD/FRED inflation series stops earlier than the other variables.

So the limiting variable at the **start** is unemployment. The limiting variable at the **end** is inflation.


In [ ]:
data.count()

`data.count()` is useful because it shows which variables are limiting our sample.

If one series ends much earlier than the others, adding it to the regression can cause us to throw away otherwise useful recent observations. This is especially important for our Colombia question because the sharp rise in consumer confidence occurs in **2025–26**.


In [ ]:
model_data.tail()

### Compare the variables

These variables have very different units, so plotting them all on the same axis is not especially useful. A correlation matrix gives us a first look at how they move together.

Remember: **correlation is not causation**. We are using these relationships to make predictions, not yet making causal claims.


In [ ]:
model_data[
    ["Consumer Confidence", "Inflation", "Unemployment", "Interest Rate", "Pesos per Dollar", "Stock Market"]
].corr().round(2)

**Questions**

1. Which variable has the strongest correlation with consumer confidence?
2. Are the signs what you expected?
3. Which explanatory variables are strongly correlated with each other?
4. Why might inflation and interest rates move together?


## 4. Can macroeconomic conditions predict the vibes?

We now build a deliberately simple model using **statsmodels**.

We will train the model using observations **before 2020**. Then we will ask it to predict consumer confidence from 2020 onward.

This gives us a useful thought experiment:

> If the pre-COVID relationship between the macroeconomy and confidence had continued, what would consumer confidence have looked like after 2020?


In [ ]:
import statsmodels.api as sm

In [ ]:
features = [
    "Inflation",
    "Unemployment",
    "Interest Rate",
    "Pesos per Dollar",
    "Stock Market"
]

target = "Consumer Confidence"

train = model_data[model_data["observation_date"] < "2020-01-01"].copy()
test = model_data[model_data["observation_date"] >= "2020-01-01"].copy()

len(train), len(test)

In [ ]:
y_test = test[target]

In [ ]:
X_train = train[features]
X_train = sm.add_constant(X_train, has_constant="add")

y_train = train[target]

model = sm.OLS(y_train, X_train).fit()

model.summary()

### What does the model think matters?

The statsmodels regression table gives us coefficients, standard errors, t-statistics, p-values, and R² in one place.

Be careful comparing coefficient sizes because the variables use different units. The signs and statistical significance are useful to inspect.


In [ ]:
model.params

Before looking at the graph, make a prediction:

**Do you expect actual Colombian consumer confidence after 2020 to be mostly above or below the model's prediction?**


### First, look at the model over the whole available period

Before zooming in on the post-COVID test period, it is useful to see the fitted relationship over the entire merged dataset.

The observations before 2020 are the **training period**. The model learned its coefficients from those observations. The observations from 2020 onward were not used to estimate the model.


### Predict the test set

The model was estimated using the pre-2020 training data. Now we apply those coefficients to the post-2020 observations.

This fills a new `Predicted Confidence` column in the `test` dataframe. We will use that column for the post-COVID plot and for the vibe-gap calculation.

In [ ]:
X_test = test[features]
X_test = sm.add_constant(X_test, has_constant="add")

test = test.copy()
test["Predicted Confidence"] = model.predict(X_test)

test[[
    "observation_date",
    "Consumer Confidence",
    "Predicted Confidence"
]].head()

In [ ]:
X_all = model_data[features]
X_all = sm.add_constant(X_all, has_constant="add")

model_data["Predicted Confidence"] = model.predict(X_all)

plt.figure(figsize=(12, 5))

plt.plot(
    model_data["observation_date"],
    model_data["Consumer Confidence"],
    label="Actual"
)

plt.plot(
    model_data["observation_date"],
    model_data["Predicted Confidence"],
    label="Predicted"
)

plt.axvline(pd.Timestamp("2020-01-01"), linestyle="--", label="Train / test split")
plt.axhline(0, linewidth=1)

plt.title("Consumer Confidence: Actual vs. Predicted")
plt.ylabel("Percentage balance")
plt.xlabel("")
plt.legend()
plt.show()

Now zoom in on the post-2020 observations. This is the true **out-of-sample test period**: the model is using relationships learned from the pre-2020 economy to predict confidence after COVID.


In [ ]:
plt.figure(figsize=(12, 5))

plt.plot(
    test["observation_date"],
    test["Consumer Confidence"],
    label="Actual"
)

plt.plot(
    test["observation_date"],
    test["Predicted Confidence"],
    label="Predicted"
)

plt.axhline(0, linewidth=1)
plt.title("Actual vs. Predicted Consumer Confidence")
plt.ylabel("Percentage balance")
plt.xlabel("")
plt.legend()
plt.show()

## 5. The vibe gap

We define the **vibe gap** as:

\[
\text{Vibe Gap} = \text{Actual Confidence} - \text{Predicted Confidence}
\]

A negative vibe gap means consumers were **more pessimistic** than the macro model predicted.

A positive vibe gap means consumers were **more optimistic** than the macro model predicted.


In [ ]:
test["Vibe Gap"] = (
    test["Consumer Confidence"] - test["Predicted Confidence"]
)

test[["observation_date", "Consumer Confidence", "Predicted Confidence", "Vibe Gap"]].tail()

In [ ]:
plt.figure(figsize=(12, 5))

plt.plot(
    test["observation_date"],
    test["Vibe Gap"]
)

plt.axhline(0, linewidth=1)
plt.title("Colombia's Vibe Gap")
plt.ylabel("Actual minus predicted confidence")
plt.xlabel("")
plt.show()

### Find the biggest vibe gaps

Which months were much more pessimistic or optimistic than the model would have expected?


In [ ]:
test.nsmallest(10, "Vibe Gap")[
    ["observation_date", "Consumer Confidence", "Predicted Confidence", "Vibe Gap"]
]

In [ ]:
test.nlargest(10, "Vibe Gap")[
    ["observation_date", "Consumer Confidence", "Predicted Confidence", "Vibe Gap"]
]

## 6. What might the model be missing?

A macroeconomic model cannot directly observe everything people are reacting to.

In Colombia, especially interesting episodes include:

- the COVID shock in 2020
- the proposed tax reform, national strike, protests, blockades, and shortages in 2021
- the 2022 presidential election and changes in expectations about economic policy
- the 2022–23 inflation shock
- debates over tax, health, pension, and labor reforms
- changes in household credit, real wages, remittances, and commodity income
- changes in perceptions of security and political stability

This is an important point about the word *vibecession*. A residual does not necessarily mean consumers are being irrational. It may mean our economic model is leaving out things that matter to households.


### Two Colombian puzzles

Use the model and the historical context to think about two different questions.

#### Puzzle 1: Why was confidence so weak around 2022–23?

Look at inflation, unemployment, interest rates, the exchange rate, and the vibe gap. Which parts of the pessimism seem to be explained by measurable macroeconomic conditions? What might be left over?

#### Puzzle 2: Why did confidence rise so strongly in 2025–26?

Return to the full consumer-confidence graph at the beginning of the notebook. The confidence series rises well beyond the endpoint of our current inflation dataset.

What additional data would you want before trying to explain the 2025–26 increase?


## 7. Extension: improve the model

Our first model is intentionally small.

Possible extensions include:

- replace the OECD inflation series with current monthly CPI data directly from **DANE**
- use the Banco de la República policy rate instead of the interbank rate
- add a monthly measure of economic activity
- add real wages or the real minimum wage
- add remittances
- add consumer credit
- add coffee prices or other commodity-income measures
- test lags: perhaps today's confidence responds to inflation or unemployment from the previous month
- compare linear regression with a Random Forest

A particularly interesting question is whether the model that explains **pessimism in 2022–23** also explains **optimism in 2025–26**.


## 8. Optional extension: original Colombian sources

The core notebook stops here with the clean FRED/OECD workflow.

For a later version, we can replace the truncated inflation series with current **DANE** CPI data and add coffee exports from the **Federación Nacional de Cafeteros**. Both official sources publish Excel workbooks rather than tidy permanent CSV endpoints, so they require an extra cleaning step.

That extra data-engineering work is useful, but it is separate from the main economic question in this notebook. For now, we keep it as an extension rather than making the core notebook depend on a fragile workbook layout.


### Possible next additions

Two especially interesting Colombia-specific additions are:

- **Coffee exports** — a real-economy measure tied to an important export sector
- **Current DANE inflation** — to extend the model through the 2025–26 rise in consumer confidence

A later version could ask whether adding coffee exports changes the predicted confidence series or reduces the size of the vibe gap.


## Data sources

Consumer confidence, unemployment, inflation, interest rates, exchange rates, and broad share prices are downloaded from the Federal Reserve Bank of St. Louis FRED database. The Colombia series used here are OECD series distributed through FRED.

The underlying Colombian consumer survey is produced by **Fedesarrollo**. Colombia's official Consumer Price Index is produced by **DANE**. Monthly coffee-export statistics are produced by the **Federación Nacional de Cafeteros de Colombia**.

FRED is used for the main teaching workflow because it provides consistent CSV files with no API key. The notebook also shows how to reach the original Colombian sources directly.
